# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [1]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


Note: you may need to restart the kernel to use updated packages.


### Read parameters

In [2]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [3]:
# load parameters from CSV into dict
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

### Global variables

In [4]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

# variables for hourly fairness and average weekly work hours
AVG_WEEKLY_HOURS    = float(params["avg_weekly_work_hours"])   # target avg weekly work hours
AVG_REFERENCE_WEEKS = int(params["avg_reference_weeks"])       # weeks window for average (currently unused, reserved)
#MAX_WEEKLY_HOURS    = AVG_WEEKLY_HOURS * AVG_REFERENCE_WEEKS   # OLD: 40*4 = 160h against a SINGLE week
# FIX (dead constraint c06): one week holds at most 7 shifts (c02), far below 160h,
# so the old cap could never bind and c06 silently did nothing. The cap is now a
# real user parameter (team decision: hard 56h limit per cycle week).
MAX_WEEKLY_HOURS    = float(params["max_weekly_work_hours"])   # hard cap on work hours per cycle week
MAX_FREE_DAYS_PER_WEEK = int(params["max_free_days_per_week"]) # max fully free days per active week (team revised 2 -> 3 on 2026-07-08)
MAX_CONSEC_FREE_DAYS = int(params["max_consec_free_days"])     # NEW c09: rolling cap, max free days IN A ROW (team: 3)
W_NB_WORKERS  = int(params["obj_w_workers"])   # weight: minimize active cycle weeks (across all cycles)
W_PESCH1 = int(params["obj_w_pesch1"])  # weight: Pesch1 deviation from Soll weekly hours
W_PESCH2 = int(params["obj_w_pesch2"])  # weight: Pesch2 equal shift-class proportion across cycles
W_CHANGEofCLASSES = int(params["obj_w_minChangeOfClasses"]) # weight: objective to minimize change of shift classes 
W_PESCH5 = int(params["obj_w_pesch5"])  # weight: Pesch5 minimize day-by-day changes of the shift TYPE (relabeled from Pesch3, team 2026-07-08)
W_PESCH0 = int(params["obj_w_pesch0"])  # weight: Pesch0 equal average weekly presence per cycle (from Pesch email)
W_PESCH4 = int(params["obj_w_pesch4"])  # weight: Pesch4 equal weekly presence across the weeks inside each cycle
W_PESCH7 = int(params["obj_w_pesch7"])   # weight: Pesch7 (even night-shift load across cycles)
W_PESCH6 = int(params["obj_w_pesch6"])   # weight: Pesch6 (even weekend-duty load across cycles)

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
MAX_NB_CYLCEs = int(params["max_nb_cycles"]) # number of cycles

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}



In [5]:
# basic inputs and parameters
cycles = range(1, MAX_NB_CYLCEs+1)
cycleWeeks  = range(1, MAX_CYCLE_WEEKS+1)
Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday (in line with static variable DICT_WEEKDAYS)

### Shift Objects
creating shift objects:

In [6]:

# read input data for shift definitions
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv",
                                 clone_ratio=float(params["reserve_clone_ratio"]))
shift_objects = abd.build_shift_objects(data_shiftSet)

# Reserve stand-bys (team decision 2026-07-08): every shift definition is cloned
# dynamically inside readShiftSet (see add_reserve_clones in src/functions.py):
# at least 1 clone per shiftID, target floor(reserve_clone_ratio * required staff),
# class one step less attractive than the original. This replaces the old hard-coded
# [reserveShift]: clones carry REAL clock times and real work hours, so min-rest,
# c06, Pesch1/2 and the output need no special cases for them.

# distuingish work shift from all shifts: 
#   WorkShifts are all shifts imported from the shift set file incl. reserve clones
#   the only non-work shift is the hard-coded freeDay below
WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)


# additional hard-coded shifts for stand-bys and free days
freeDayShift = Shift.Shift("[freeDay_:-)_]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           dt.time(*map(int,"07:00".split(':'))),
                           dt.time(*map(int,"07:00".split(':'))),
                           0, 
                           1, 
                           0, 
                           0, 
                           None 
                           )
shift_objects.append(freeDayShift)


#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

In [7]:
# preparation for output
shift_info = {
    s.shift_id: {
        "start": s.start.strftime("%H:%M"),
        "end": s.end.strftime("%H:%M"),
        "workingTime": s.shift_work_time_assignment
    }
    for s in shift_objects
}

In [8]:
# Pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days.
# Excludes the freeDay dummy since it has no real start/end times.
# Result: list of (sh1_id, sh2_id) tuples that cannot appear on consecutive days in a snake.
DUMMY_SHIFTS = {"[freeDay_:-)_]"}
incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]

# Night shifts by clock time (end <= start = runs past midnight). Used by Pesch7 and
# kept dynamic on purpose: reserve clones of night shifts qualify automatically, and
# min-rest above already bans night -> night-clone (9h15 gap < 11h) with no special
# case. The old explicit night -> reserve ban became obsolete when the placeholder
# [reserveShift] was replaced by clones with real times.
overnight_ids = [s.shift_id for s in shift_objects
                 if s.shift_id not in DUMMY_SHIFTS and s.end <= s.start]


In [9]:
# Pre-compute work hours per shift using shift_duration_hours from Shift.py.
#
# FIX (reserve hours, 2026-07-07): reserveShift is REAL work time (24h standby,
# the person must be ready to jump in at any moment), but it used to be excluded
# here together with freeDay via DUMMY_SHIFTS, so its hours were invisible to
# Pesch1, c06 and the output. Only the free day truly carries no hours.
# DUMMY_SHIFTS (previous cell) still excludes both from the automatic MIN_REST
# pairing, because their clock times are placeholders.
NO_HOURS_SHIFTS = {freeDayShift.shift_id}
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    #if s.shift_id not in DUMMY_SHIFTS   # OLD: also dropped reserveShift -> its 24h were lost
    if s.shift_id not in NO_HOURS_SHIFTS # FIX: only the free day has no hours
}

# net work time per shift (break excluded); fall back to presence time if not specified ('none')
work_hours = {
    s.shift_id: (float(s.shift_work_time_assignment)
                 if s.shift_work_time_assignment is not None
                 else Shift.shift_duration_hours(s))
    for s in shift_objects
    #if s.shift_id not in DUMMY_SHIFTS   # OLD: also dropped reserveShift -> its 24h were lost
    if s.shift_id not in NO_HOURS_SHIFTS # FIX: only the free day has no hours
}

#when none then gleich wie shift hours 

### modelling

initiate model and basic decision variables

$x \to$ x  
$y \to$ active\_week  
$z \to$ active\_cycle  

In [10]:
# modelling

modelCycle = gp.Model("SnakeBuilding_simple")

# --- decision variables
# x[c, s, d, sh] = 1 if in cycle c, cycleWeek s, on weekday d, shift sh is assigned
x = modelCycle.addVars(cycles, cycleWeeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all, 0 otherwise
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_week[c,s] = 1 if cycle c uses cycleWeek s (i.e., at least one shift in that week is active), 0 otherwise
active_week = modelCycle.addVars(cycles, cycleWeeks, vtype=GRB.BINARY, name="active_week")


Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2810721


Academic license 2810721 - for non-commercial use only - registered to ar___@student.uni-siegen.de


### constraints:

#### basic constraints

$ y_{c,w} - z_w <= 0$,  $ \forall c $  $ \forall w $  

In [11]:
# basic constraints

# ensure a cycleWeek can only be active when the according cylce is active
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(active_week[c,w] - active_cycle[c] <= 0,
                             name=f"WeekImpliesCycle_c{c}_s{w}")

$ \sum_{d \in D} \sum_{s \in S}{x_{c,w,d,s}} \leq |D| \cdot |S| \cdot y_{c,w}  $  ,  $ \forall c $  $ \forall w $  

In [12]:
# enforce active_week >= any assignment in that week
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, sh] for d in Weekdays for sh in Shifts) <= len(Weekdays) * len(Shifts) * active_week[c, w],
            name=f"Link_x_activeWeek_c{c}_s{w}_upper"
        )

In [13]:
# cycle must be active if any week in that cycle is active
# "no active cycleWeek without active cycle"
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) >= active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}")


ensure that cycles and cycleWeeks are activated in ascending order  

$y_s - y_{s+1} >= 0, \forall s \in \{1,\dots,M-1\}, \forall c \in C$  
$z_c - z_{c+1} >= 0, \forall c \in \{1,\dots,C-1\}$

In [14]:

for c in range(1, MAX_NB_CYLCEs): # loop from first to second-last entry
    modelCycle.addConstr(active_cycle[c] - active_cycle[c + 1] >= 0,
                         name=f"CycleOrder_c{c}")

for c in cycles: # loop across all possible cycles
    for w in range(1, MAX_CYCLE_WEEKS): # loop from first to second-last entry
        modelCycle.addConstr(active_week[c, w] - active_week[c, w + 1] >= 0,
                             name=f"WeekOrder_c{c}_s{w}")


#### condition c01

each shift has to be covered on each day exactly once

$\sum_{s=1}^{n}{x_{c,w,d,s}} >= 1$  
$ \forall d \in D,$  
$ \forall w \in N,$  
$ \forall c \in M$  

$x_{c,w,d,s} = 1$, when cycle c is working in cycleWeek w on day d, covering shift s

$x: $ binary variable,
$c: $ cycle,
$d: $ weekday,
$w: $ cycleWeek
$s: $ shift


In [15]:
for d in Weekdays: #loop over all week days
    for ws in WorkShifts: # loop over all work shift
        shift = next(s for s in shift_objects if s.shift_id == ws) # 'next()' is an alternative for 'for s in shift_objects: if s.shift_id == ws: shift = s'
#       if DICT_WEEKDAYS[shift.weekdays[0]] <= d <= DICT_WEEKDAYS[shift.weekdays[-1]]:
        if d in [DICT_WEEKDAYS[w] for w in shift.weekdays]: # correction to also cover shifts that appear for non-consecutive week days (eg. Mon, Wed)
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) >= 1,
                        name=f"Cover_day{d}_{ws}")
        else:
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) <= 0,
                        name=f"Cover_day{d}_{ws}")

#Логика: перед добавлением ограничения проверяем входит ли день d в список weekdays этой смены. Если нет ограничение не добавляется, смена в этот день не требуется.
#Logic: Before adding a restriction, we check whether day d is included in the list of weekdays for this shift. If not, the restriction is not added, as no shift is required on that day.

# 1. each shift has to be covered on each day
#for d in Weekdays:
#    for ws in WorkShifts:
#        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
#                    name=f"Cover_day{d}_{ws}")



#### condition c02

<div align="left">

$ \sum_{s \in S}{x_{c,w,d,s}} - y_s = 0$  
$ \forall c \in C, \forall w \in W, \forall d \in D $

</div>

In [16]:
# 2. each cycle shall have exactly one shift per day (implying that on cycleWeek has exactly one shift per day)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, w, d, sh] for sh in Shifts) - active_week[c, w] == 0,
                name=f"OneShiftPerDay_c{c}_s{w}_d{d}"
            )


#### condition c03

$ \sum_{d = 1}^{8-MAX\_CONSEC\_DAYS+1}{x_{}} $  
$ \forall c \in C, \forall w \in W, \forall d \in {1,\dots, 8 - MAX\_CONSEC\_DAYS +1} $

In [17]:
# c03: max MAX_CONSEC_DAYS consecutive working days (reviewed 2026-07-09: correct).
# Formulation: every sliding window of MAX_CONSEC_DAYS + 1 days on the global timeline
# contains at most MAX_CONSEC_DAYS work days = no 6 consecutive work days anywhere,
# including across week borders. The ring cell extends the same windows across the wrap.
# OPEN INTERPRETATION QUESTIONS for the team (not bugs, the spec is silent):
#   - is a minimum rest required AFTER a full 5-day block? (currently: no extra rest)
#   - is the requirement "max 5 IN A ROW" (implemented) or "max 5 within any 7 days"?

# Helper function: convert a global day index t into tupel (cycleWeek, weekday)
# This allows to treat all days across all cycleWeeks as one continuous timeline.
def decode_global_day(t):
    s = (t - 1) // 7 + 1 # Compute cycleWeek index from global day t (1-based indexing)
    d = (t - 1) % 7 + 1 # Compute weekday index from global day t (1-based indexing)
    return s, d

# total number of days across all cycleWeeks
# used to define the global timeline over which consecutive work days are checked.
TOTAL_DAYS = MAX_CYCLE_WEEKS * 7

# limit the number of consecutive working days across all cycleWeeks
# for each cycle scan through the entire global timeline and check every (MAX_CONSEC_DAYS + 1)-days window
# to ensure that not all days in the window are working days.
for c in cycles:

    # iterate over all possible start positions of a sliding window
    # the window length is MAX_CONSEC_DAYS + 1, so the last valid start is:
    # TOTAL_DAYS - MAX_CONSEC_DAYS
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_DAYS + 1):

        # Collect expressions representing work indicators for each day in the window
        moving_time_window = []

        # iterate through each offset inside the window
        # offset = 0 means the first day of the window
        # offset = MAX_CONSEC_DAYS means the last day of the window
        for offset in range(0, MAX_CONSEC_DAYS + 1):

            # Compute the global day index inside the window
            t = t_start + offset

            # Convert global day index back to (cycleWeek, weekday)
            s, d = decode_global_day(t)

            # work[c,s,d] = sum of all work shifts assigned on that day
            # If any WorkShift is assigned, this sum becomes 1 (binary model)
            moving_time_window.append(
                gp.quicksum(x[c, s, d, sh] for sh in WorkShifts)
            )

        # ACTUAL CONSTRAINT COMES HERE:
        # In any time window of length MAX_CONSEC_DAYS + 1,
        # the number of working days must be <= MAX_CONSEC_DAYS.
        # This prevents sequences of MAX_CONSEC_DAYS + 1 consecutive working days.
        modelCycle.addConstr(
            gp.quicksum(moving_time_window) <= MAX_CONSEC_DAYS,
            name=f"MaxConsecDays_c{c}_t{t_start}"
        )


#### Condition c05

In [18]:
# c05: minimum rest time between consecutive shifts within a snake week.
# If sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same snake.
# Covers days 1-6 only; wrap-around (day 7 -> day 1) not yet modelled. => DONE NOW
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:  # for Mon to Sat use pairs of day d and d+1
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w, d+1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            elif d == 7 and w < MAX_CYCLE_WEEKS:  # Sun of week w -> Mon of NEXT week w+1 (snake runs continuously)
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w+1, 1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            # FIX (C05 wrap-around): the Sun->Mon transition below originally paired
            # Sunday of week w with Monday of the SAME week w. In a snake the weeks run
            # continuously, so Sunday of week w is followed by Monday of the NEXT week (w+1).
            # The old version constrained a non-existent backward pair and left the real
            # week-to-week rest gap unchecked. Now paired with w+1, guarded by w < MAX_CYCLE_WEEKS
            # (the last week has no successor). Old code kept commented below for reference.          

#### Condition c06

In [19]:
# c06: total work hours per active cycle week must not exceed MAX_WEEKLY_HOURS.
# NOTE (2026-07-07): MAX_WEEKLY_HOURS is now the hard 56h user parameter (see the
# globals cell). The old value (40h * 4 weeks = 160h) was applied to a SINGLE week
# and could never be reached, so this constraint used to be permanently slack.
# ensures no cycle week accumulates more than the allowed weekly work time.
# bound scales with active[s] so inactive cycles are not constrained.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(
               #shift_hours.get(sh, 0) * x[c, w, d, sh] # replaced by more efficient version without "get()":
                work_hours[sh] * x[c, w, d, sh]
                for d in Weekdays
               #for sh in Shifts if sh in shift_hours # # replaced by more efficient version:
                for sh in work_hours.keys()
            ) <= MAX_WEEKLY_HOURS * active_week[c, w],
            name=f"MaxWeeklyHours_c{c}_w{w}"
        )

#### Condition c07  
ensure balanced [almost equal] lenght of cycles (same number of cycleWeeks plus/minus 1 week)

$ W_c = \sum_{w=1}^{M} y_{c,w} \text{  : counting active weeks } w \text{ within cycle } c$  

$M: \text{MAX\_CYCLE\_WEEKS}$  

$\text{condition: }|W_c - W_{c'}| \le 1 \quad \forall c,c' \in C$  

$\text{linearized condition: }$  
$W_c - W_{c'} \le 1 \quad \forall c,c' \in C$  
$W_{c'} - W_c \le 1 \quad \forall c,c' \in C$

In [20]:
# c07: ensure all used cycles have balanced number of cycleWeeks;
# deviations of at most 1 week are allowed

# determine number of active weeks per cycle
# (W_c in formula)
cycle_length = {c: gp.quicksum(active_week[c, s] for s in cycleWeeks) for c in cycles}

# pairwise balance constraints
# |W_c1 - W_c2| <= 1  for all cycles c1 != c2 => implemented as 2 linear statements
for c1 in cycles:
    for c2 in cycles:
        if c1 < c2:  # avoiding duplicates and self-pairing
            # W_c1 - W_c2 <= 1
            modelCycle.addConstr(
                cycle_length[c1] - cycle_length[c2] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c2]), name=f"CycleBalance_upper_c{c1}_c{c2}") ## upper bound: cycle c1 cannot be more than 1 week longer than c2 — relaxed if c2 is inactive
            # W_c2 - W_c1 <= 1
            modelCycle.addConstr(cycle_length[c2] - cycle_length[c1] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c1]), name=f"CycleBalance_lower_c{c1}_c{c2}") ## lower bound: cycle c2 cannot be more than 1 week longer than c1 — relaxed if c1 is inactive
# Big-M: if cycle c is inactive (active_cycle=0), the right side becomes 1+MAX_CYCLE_WEEKS, which is always larger than any possible difference — so the constraint has no effect

#### condition c08
max fully free days per active week (team decision 2026-07-07)

In [21]:
# c08: at most MAX_FREE_DAYS_PER_WEEK completely free days per active week.
# TEAM REVISION (2026-07-08): cap raised from 2 to 3 (parameter) and complemented
# by c09 below, a ROLLING cap on consecutive free days. The two answer different
# concerns: c08 keeps every week a real working week (floor on workdays), c09
# stops long idle stretches that c08 alone allows across week seams
# (e.g. Sat+Sun free + Mon+Tue free = 4 in a row, legal for c08).
# Team decision (2026-07-07): a snake week must stay a real working week; a worker
# gets at most 2 fully free days per week ("free" = not even standby duty; the
# reserveShift does NOT count here, it is work).
# Known and intended interaction: a night shift can only be followed by a free day
# (min rest kills night->day and night->night, and the team banned night->reserve),
# so this cap indirectly limits nights to about 3 per week and makes "pure night
# weeks" impossible. That is deliberate: the load should stay mixed and fair.
# For inactive weeks the right-hand side is 0 and c02 already forces all x to 0.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, freeDayShift.shift_id] for d in Weekdays)
            <= MAX_FREE_DAYS_PER_WEEK * active_week[c, w],
            name=f"MaxFreeDays_c{c}_w{w}"
        )

In [22]:
# Pesch1: deviation of average weekly net work time from target (AVG_WEEKLY_HOURS)
#cycle_hours = {
#    c: gp.quicksum(work_hours[sh] * x[c, w, d, sh]
#                   for w in cycleWeeks for d in Weekdays for sh in work_hours.keys())
#    for c in cycles
#}
#dev_pos = modelCycle.addVars(cycles, lb=0, name="dev_pos")
#dev_neg = modelCycle.addVars(cycles, lb=0, name="dev_neg")
#
#for c in cycles:
#    modelCycle.addConstr(
#        cycle_hours[c] - AVG_WEEKLY_HOURS * cycle_length[c] == dev_pos[c] - dev_neg[c],
#        name=f"Pesch1_dev_c{c}"
#    )

#####
# Pesch1: per-week deviation of net work time from the weekly target (AVG_WEEKLY_HOURS).
# Soft target: the objective penalises deviation, it does NOT force exactly 40h.
# Measured per week so heavy/light weeks cannot cancel out across the cycle.
week_hours = {
    (c, w): gp.quicksum(work_hours[sh] * x[c, w, d, sh]
                        for d in Weekdays for sh in work_hours.keys())
    for c in cycles for w in cycleWeeks
}

dev_pos = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_pos")
dev_neg = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_neg")

for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            week_hours[(c, w)] - AVG_WEEKLY_HOURS * active_week[c, w] == dev_pos[c, w] - dev_neg[c, w],
            name=f"Pesch1_dev_c{c}_w{w}"
        )



In [23]:
# Pesch2: equal proportion of shift categories (by presence time) across cycles.
# Category = shift_class value. Uses shift_hours (presence / Anwesenheitszeit), NOT work_hours.
#shift_class_of = {s.shift_id: s.shift_class for s in shift_objects if s.shift_id not in DUMMY_SHIFTS}  # OLD: dropped reserve
# FIX (2026-07-07): shift_hours now includes reserveShift (it is real work), so the
# class map must cover it too, otherwise the cat_hours lookup below crashes.
# reserve currently shares class 5 with the weekend day shift (open team question).
shift_class_of = {s.shift_id: s.shift_class for s in shift_objects if s.shift_id not in NO_HOURS_SHIFTS}
categories = sorted(set(shift_class_of.values()))

# presence hours of each category k inside each cycle c
cat_hours = {
    (c, k): gp.quicksum(shift_hours[sh] * x[c, w, d, sh]
                        for w in cycleWeeks for d in Weekdays
                        for sh in shift_hours.keys() if shift_class_of[sh] == k)
    for c in cycles for k in categories
}

# ---- Pesch2 v2 (2026-07-07): each cycle vs the GLOBAL class mix -------------
# The old pairwise version (hashed below) only balanced neighbouring cycles
# (1~2, 2~3), so the outer cycles could drift apart by twice the tolerance, and
# it needed a Big-M (grown to ~3000 after reserve got hours) that hurt numerics.
# The spec formula instead compares each cycle's class share with the share p_k
# of that class in the WHOLE dataset. p_k is a plain data constant, so
# p_k * total_pres stays linear, and an inactive cycle gives 0 == 0 by itself:
# no Big-M needed at all.

# p_k: share of class k in the dataset's coverage-required presence hours
dataset_class_hours = {k: 0.0 for k in categories}
for s in shift_objects:
    if s.shift_id in shift_class_of:
        dataset_class_hours[shift_class_of[s.shift_id]] += shift_hours[s.shift_id] * len(s.weekdays)
p = {k: dataset_class_hours[k] / sum(dataset_class_hours.values()) for k in categories}

# total presence hours of each cycle (sum of its category hours)
total_pres = {c: gp.quicksum(cat_hours[c, k] for k in categories) for c in cycles}

d2_pos = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_pos")
d2_neg = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_neg")

# deviation of class k in cycle c from its fair share of that cycle's hours
for c in cycles:
    for k in categories:
        modelCycle.addConstr(
            cat_hours[c, k] - p[k] * total_pres[c] == d2_pos[c, k] - d2_neg[c, k],
            name=f"Pesch2_share_c{c}_k{k}"
        )

# OLD pairwise version, kept for reference:
## Balance category hours between consecutive cycles (c04 chains them: c1 ~ c2 ~ c3).
## Soft: deviation goes into the objective. Big-M relaxes the pair when the higher
## cycle is inactive (same trick as c07), so we only balance actually-used cycles.
#pair_cycles = range(1, MAX_NB_CYLCEs)  # pairs (c, c+1)
#d2_pos = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_pos")
#d2_neg = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_neg")

#BIG_M_HOURS = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())

#for c in pair_cycles:
#    for k in categories:
#        diff = cat_hours[c, k] - cat_hours[c+1, k]
#        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) <=  BIG_M_HOURS * (1 - active_cycle[c+1]),
#                             name=f"Pesch2_bal_up_c{c}_k{k}")
#        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) >= -BIG_M_HOURS * (1 - active_cycle[c+1]),
#                             name=f"Pesch2_bal_lo_c{c}_k{k}")

#### Pesch3 (a.k.a. Pesch8) / minimize day-to-day changes of shift class  

$$ \sum_{c=1}^{M}{x_{c,w,d,s}},\text{ } \forall w,d,s$$

In [24]:
# pre-work for condition to have as less shift class changes as possible (target: same shift-class in blocks as far as possible)

# Map each shift id to its shift_class (eg 'dayShift': 1)
dict_shift_to_class = {s.shift_id: s.shift_class for s in shift_objects}

# unique list of shift classes from shift definition (from input file)
set_classes = sorted(set(dict_shift_to_class.values()))


In [25]:
# additional decision variables to link shift classes to active shifts/shiftWeek/weekDay/cycle:
# NOTE (2026-07-09): these helper variables are declared CONTINUOUS [0,1] on purpose.
# assigned-vars are equality-linked to the binary x (so they are 0/1 automatically),
# diff-vars only have lower bounds and are minimized in the objective (so they settle
# at the exact |difference|). Removing them from branching speeds up the MIP.

# binary variable indicating whether a specific class is assigned on (c,w,d), ie in cycle c, in cycleWeek w, on weekDay d
# class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k
class_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="class_assigned")

# add supporting variables for absolute differences per class between consecutive days
# diff[c,w,d,k] >= | class_assigned[c,w,d,k] - class_assigned[c,w,d+1,k] |
class_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="diff_class")


In [26]:
# link class_assigned to x
# for each shiftClass k, class_assigned equals the sum of x over all shifts that belong to that class
# this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
dict_shifts_by_class = {k: [sh for sh, shiftClass in dict_shift_to_class.items() if shiftClass == k and sh in WorkShifts] for k in set_classes}
# results in sth like this:
    # 2: ["[00day000week]%_%1", "[00day000week]%_%2", ...],
    # 5: ["[00dayweekend]%_%1", ...],
    # 7: ["[night000week]%_%1", "[night000week]%_%2", ...],
    # 9: ["[nightweekend]%_%1", ...],
    #10: ["[allDay_shift]%_%1"]
    # => focus on WorkShifts - ie excluding free days - is crucial here: otherwise the solver will add arbitrarily free days

for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            for k in set_classes: 
                shiftClass = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
                # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
                # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
                if shiftClass: # run the following code only for non-empty lists of shifts in class k
                    modelCycle.addConstr(
                        class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in shiftClass) == 0,
                        name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
                    )
                else:
                    # if no shifts for this class (shouldn't happen by definition), force 0 (just a safety net)
                    modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
                                         name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# TASK: to be added: loop from last active week to first day of first week
for c in cycles:
    #for w in range(1, MAX_CYCLE_WEEKS-1):   # OLD BUG: stopped two weeks early, so the
    #                                        # transitions of the last two weeks were never
    #                                        # counted and the solver could hide class chaos there
    for w in cycleWeeks:                     # FIX: cover every week
        for d in Weekdays:
            # determine next day index (wrap 7 -> 1)
            if d <= 6:
                d_next = d + 1
                w_next = w
            #else:  # d == 7                          # OLD: no guard, w+1 exploded past the last index
            elif w < MAX_CYCLE_WEEKS:  # d == 7: Sunday -> Monday of the NEXT week
                d_next = 1
                w_next = w+1  # use first day of next cycleWeek
            else:
                continue  # Sunday of the very last week: there is no next week here;
                          # the wrap back to week 1 is handled by the RING CLOSURE cell below
            for k in set_classes:
                # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
                modelCycle.addConstr(
                    #class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],  # OLD: ungated
                    # FIX (gating): '- (1 - active_week[...])' switches the constraint off when the
                    # next day's week is inactive. Without it, stepping from the last active week
                    # into the empty rest of the cycle was counted as a phantom class change.
                    class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k] - (1 - active_week[c, w_next]),
                    name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
                )
                # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
                modelCycle.addConstr(
                    #class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],  # OLD: ungated
                    class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k] - (1 - active_week[c, w_next]),
                    name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
                )
# Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.

class_changes_total = 0.5 * gp.quicksum(class_diff[c, w, d, k] for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)

In [27]:
### seems to be duplicated cell ???
# # link class_assigned to x
# # for each class k, class_assigned equals the sum of x over all shifts that belong to that class
# # this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
# dict_shifts_by_class = {k: [sh for sh, cls in dict_shift_to_class.items() if cls == k] for k in set_classes}

# for c in cycles:
#     for w in cycleWeeks:
#         for d in Weekdays:
#             for k in set_classes:
#                 class_shifts = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
#                 # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
#                 # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
#                 if class_shifts:
#                     modelCycle.addConstr(
#                         class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in class_shifts) == 0,
#                         name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
#                     )
#                 else:
#                     # if no shifts for this class (shouldn't happen by definition), force 0
#                     modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
#                                          name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# # class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# # TASK: to be added: loop from last active week to first day of first week
# for c in cycles:
#     for w in range(1, MAX_CYCLE_WEEKS-1):
#         for d in Weekdays:
#             # determine next day index (wrap 7 -> 1)
#             if d <= 6:
#                 d_next = d + 1
#                 w_next = w
#             else:  # d == 7
#                 d_next = 1
#                 w_next = w+1  # use first day of next cycleWeek
#             for k in set_classes:
#                 # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],
#                     name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
#                 )
#                 # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],
#                     name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
#                 )
# # Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# # When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.


#### Ring closure (the snake is a cycle, not a line)
Spec: "the last shift in the snake and the shift at the very first snake position must also observe their required time constraints". After the last active week the worker starts again at week 1, so Sunday of the last active week is followed by Monday of week 1. The three day-by-day rules (c05 min rest, c03 max consecutive working days, Pesch8 class changes) are closed across that seam here.

In [28]:
# ============================== RING CLOSURE ==============================
# WHY THIS EXISTS: all constraints so far treat the cycle as a straight LINE of
# weeks and simply stop at the last one. In reality the rotation wraps around:
# after the last active week the worker continues with week 1. So the pair
# (Sunday of the LAST ACTIVE week) -> (Monday of week 1) is a real consecutive
# pair of days and must obey the same rules as any other day pair.
#
# THE ONE TRICK USED BY EVERY BLOCK BELOW:
# "is week w the LAST ACTIVE week of cycle c?"
# Weeks activate as a prefix without holes (WeekOrder): 1,1,...,1,0,...,0.
# Therefore   active_week[c,w] - active_week[c,w+1]
#   = 1 exactly at the boundary (1 followed by 0)  -> w IS the last active week
#   = 0 everywhere else (1-1 inside the prefix, 0-0 after it).
# For the very last index there is no w+1, so the indicator is active_week[c,MAX]
# itself. This is a plain linear expression built from EXISTING variables, so no
# new binaries are needed. Each ring constraint is written so that it is active
# when the indicator is 1 and trivially satisfied (relaxed) when it is 0.
def last_active_expr(c, w):
    # 0/1-valued linear expression: 1 iff week w is the last active week of cycle c
    if w < MAX_CYCLE_WEEKS:
        return active_week[c, w] - active_week[c, w + 1]
    return active_week[c, w]

# ---- (a) c05 ring: minimum rest across the wrap ---------------------------
# For every incompatible pair (sh1, sh2): if sh1 sits on the last active Sunday,
# sh2 may not sit on Monday of week 1.
#   x1 + x2 <= 2 - last_active(c,w)
# If w is the last active week the bound is 1 (the usual "not both allowed").
# Otherwise the bound is >= 2 and two binaries can never exceed it -> no effect.
for c in cycles:
    for w in cycleWeeks:
        for (sh1, sh2) in incompatible_pairs:
            modelCycle.addConstr(
                x[c, w, 7, sh1] + x[c, 1, 1, sh2] <= 2 - last_active_expr(c, w),
                name=f"RingMinRest_c{c}_w{w}_{sh1}_{sh2}"
            )

# ---- (b) c03 ring: max consecutive working days across the wrap -----------
# A forbidden run of MAX_CONSEC_DAYS+1 working days can also straddle the seam:
# k days at the END of the last active week + (MAX_CONSEC_DAYS+1-k) days at the
# START of week 1. We enumerate every split k = 1..MAX_CONSEC_DAYS.
# When w is NOT the last active week, the right side is inflated by the full
# window length, which makes the constraint impossible to violate there.
for c in cycles:
    for w in cycleWeeks:
        for k in range(1, MAX_CONSEC_DAYS + 1):
            tail = gp.quicksum(x[c, w, d, sh]                     # last k days of week w
                               for d in range(8 - k, 8) for sh in WorkShifts)
            head = gp.quicksum(x[c, 1, d, sh]                     # first (MAX+1-k) days of week 1
                               for d in range(1, MAX_CONSEC_DAYS + 2 - k) for sh in WorkShifts)
            modelCycle.addConstr(
                tail + head <= MAX_CONSEC_DAYS
                             + (MAX_CONSEC_DAYS + 1) * (1 - last_active_expr(c, w)),
                name=f"RingMaxConsec_c{c}_w{w}_k{k}"
            )

# ---- (c) Pesch8 ring: class change across the wrap ------------------------
# One extra transition per cycle: last active Sunday -> Monday of week 1.
# We REUSE the existing class_diff[c, w, 7, k] variables. This cannot clash with
# their normal use (Sunday -> Monday of week w+1), because that constraint is
# gated OFF exactly when week w+1 is inactive, i.e. exactly when w is the last
# active week. The two uses are mutually exclusive by construction.
for c in cycles:
    for w in cycleWeeks:
        for k in set_classes:
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, w, 7, k] - class_assigned[c, 1, 1, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffPos_c{c}_w{w}_k{k}"
            )
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, 1, 1, k] - class_assigned[c, w, 7, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffNeg_c{c}_w{w}_k{k}"
            )

#### Pesch5: minimize day-by-day changes of the shift TYPE (relabeled from Pesch3, team 2026-07-08)
Twin of Pesch8: same machinery, different grouping key (shift type = base shiftID instead of shift class).

In [29]:
# Pesch5 (team relabel 2026-07-08, formerly Pesch3): try to keep the SAME shift (by shiftID) on
# NOTE (2026-07-09): these helper variables are declared CONTINUOUS [0,1] on purpose.
# assigned-vars are equality-linked to the binary x (so they are 0/1 automatically),
# diff-vars only have lower bounds and are minimized in the objective (so they settle
# at the exact |difference|). Removing them from branching speeds up the MIP.
# consecutive days. Twin of Pesch8 above; the only difference is the grouping key:
#   Pesch8 groups by shift_class -> "stay in the same comfort category"
#   Pesch5 groups by shift TYPE  -> "stay on the exact same shift"
# With the current data (almost one class per type) both behave similarly; they
# diverge once several types share one class (e.g. all day shifts = one class):
# day-week -> day-weekend is then fine for Pesch3/8 (class) but still a change for Pesch5.
# freeDay carries no type (same WorkShifts filter as Pesch8), so entering/leaving
# a free day costs half a change; a free block costs the same as a single free day.
# Merging the class and type builders into one is refactoring work (S8), done with Dirk.

# type = base shift name without the %_% copy suffix, e.g. "[00day000week]"
dict_shift_to_type = {sh: sh.split("%_%", 1)[0] for sh in WorkShifts}
set_types = sorted(set(dict_shift_to_type.values()))
dict_shifts_by_type = {t: [sh for sh, tt in dict_shift_to_type.items() if tt == t] for t in set_types}

# type_assigned[c,w,d,t] = 1 iff the shift worked on (c,w,d) has type t
type_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="type_assigned")
# type_diff >= |type_assigned(day) - type_assigned(next day)| per type
type_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="type_diff")

# link type_assigned to x (same pattern as LinkClass in Pesch8)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            for t in set_types:
                modelCycle.addConstr(
                    type_assigned[c, w, d, t] - gp.quicksum(x[c, w, d, sh] for sh in dict_shifts_by_type[t]) == 0,
                    name=f"LinkType_c{c}_w{w}_d{d}_{t}"
                )

# count type changes between consecutive days; gating and week handling mirror the
# FIXED Pesch8 loop (all weeks covered, inactive next week switches the pair off)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:
                d_next, w_next = d + 1, w
            elif w < MAX_CYCLE_WEEKS:
                d_next, w_next = 1, w + 1
            else:
                continue  # wrap of the very last week is handled by the ring part below
            for t in set_types:
                modelCycle.addConstr(
                    type_diff[c, w, d, t] >= type_assigned[c, w, d, t] - type_assigned[c, w_next, d_next, t]
                                             - (1 - active_week[c, w_next]),
                    name=f"TypeDiffPos_c{c}_w{w}_d{d}_{t}"
                )
                modelCycle.addConstr(
                    type_diff[c, w, d, t] >= type_assigned[c, w_next, d_next, t] - type_assigned[c, w, d, t]
                                             - (1 - active_week[c, w_next]),
                    name=f"TypeDiffNeg_c{c}_w{w}_d{d}_{t}"
                )

# ring: one wrap transition per cycle (last active Sunday -> Monday of week 1),
# same last_active_expr trick and the same variable-reuse argument as Pesch8's ring
for c in cycles:
    for w in cycleWeeks:
        for t in set_types:
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, w, 7, t] - type_assigned[c, 1, 1, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffPos_c{c}_w{w}_{t}"
            )
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, 1, 1, t] - type_assigned[c, w, 7, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffNeg_c{c}_w{w}_{t}"
            )

# one real change flips two types (1->0 and 0->1), hence the 0.5 factor
type_changes_total = 0.5 * gp.quicksum(type_diff[c, w, d, t]
                                       for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)

#### condition c09
rolling cap: no more than MAX_CONSEC_FREE_DAYS free days in a row (team revision, 2026-07-08)

In [30]:
# c09: at most MAX_CONSEC_FREE_DAYS free days IN A ROW, measured on the rolling
# day timeline, not per calendar week. Complements c08 (see comment there): it
# closes the week-seam hole where two legal weeks still produce a long idle block.
# Same sliding-window idea as c03: in every window of MAX_CONSEC_FREE_DAYS + 1
# consecutive days, at least one day must NOT be a free day.
# Inactive weeks carry no assignments at all (c02), so windows reaching into the
# inactive tail are satisfied automatically.
FREE_ID = freeDayShift.shift_id
for c in cycles:
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_FREE_DAYS + 1):
        window = []
        for offset in range(0, MAX_CONSEC_FREE_DAYS + 1):
            s, d = decode_global_day(t_start + offset)
            window.append(x[c, s, d, FREE_ID])
        modelCycle.addConstr(
            gp.quicksum(window) <= MAX_CONSEC_FREE_DAYS,
            name=f"MaxConsecFree_c{c}_t{t_start}"
        )

# ring part: idle runs can also straddle the wrap (end of the LAST active week ->
# start of week 1). Same split enumeration and the same last_active_expr gating
# as the RingMaxConsec block above, just counting free days instead of work days.
for c in cycles:
    for w in cycleWeeks:
        for k in range(1, MAX_CONSEC_FREE_DAYS + 1):
            tail = gp.quicksum(x[c, w, d, FREE_ID] for d in range(8 - k, 8))
            head = gp.quicksum(x[c, 1, d, FREE_ID] for d in range(1, MAX_CONSEC_FREE_DAYS + 2 - k))
            modelCycle.addConstr(
                tail + head <= MAX_CONSEC_FREE_DAYS
                             + (MAX_CONSEC_FREE_DAYS + 1) * (1 - last_active_expr(c, w)),
                name=f"RingMaxConsecFree_c{c}_w{w}_k{k}"
            )

#### Pesch0 + Pesch4: presence balance
Pesch0 (from the professor's email, was missing on the board): equal average weekly presence per Turnus, i.e. balance BETWEEN cycles. Pesch4: equal weekly presence across the weeks INSIDE each Turnus(gruppe).

In [31]:
# Both criteria are about PRESENCE time (Anwesenheit), so shift_hours is used.
# pres_week[c,w] = gross presence hours of week w in cycle c
pres_week = {
    (c, w): gp.quicksum(shift_hours[sh] * x[c, w, d, sh] for d in Weekdays for sh in shift_hours)
    for c in cycles for w in cycleWeeks
}
MAX_WEEK_PRESENCE = 7 * 24   # small gating constant (168h), not a huge Big-M

# ---- Pesch4: equal weekly presence WITHIN each cycle (Turnusgruppe) ---------
# Spec formula: min max |W(a_i) - W(a_j)| inside one snake. Standard linearization:
# p4max[c] sits above every active week, p4min[c] sits below every active week,
# the objective pays for the spread p4max - p4min. The (1 - active_week) terms
# gate inactive weeks out; p4min <= p4max pins inactive cycles to spread 0.
p4max = modelCycle.addVars(cycles, lb=0, name="pesch4_max")
p4min = modelCycle.addVars(cycles, lb=0, name="pesch4_min")
for c in cycles:
    modelCycle.addConstr(p4min[c] <= p4max[c], name=f"Pesch4_anchor_c{c}")
    for w in cycleWeeks:
        modelCycle.addConstr(
            p4max[c] >= pres_week[(c, w)] - MAX_WEEK_PRESENCE * (1 - active_week[c, w]),
            name=f"Pesch4_up_c{c}_w{w}"
        )
        modelCycle.addConstr(
            p4min[c] <= pres_week[(c, w)] + MAX_WEEK_PRESENCE * (1 - active_week[c, w]),
            name=f"Pesch4_lo_c{c}_w{w}"
        )

# ---- Pesch0: equal AVERAGE weekly presence ACROSS cycles --------------------
# The exact average needs total / length, i.e. division by a variable: nonlinear.
# Honest proxy: c07 already keeps cycle lengths within +-1 week, so equal TOTALS
# equal averages up to at most one week of presence. Good enough as a soft goal.
# Pairs are gated with Gurobi INDICATOR constraints instead of Big-M, so nothing
# ugly enters the coefficient matrix: a pair only counts while the higher cycle
# is active (c04 ordering then guarantees the lower one is active too).
cycle_pairs = [(c1, c2) for c1 in cycles for c2 in cycles if c1 < c2]
p0_dev = modelCycle.addVars(cycle_pairs, lb=0, name="pesch0_dev")
for (c1, c2) in cycle_pairs:
    modelCycle.addGenConstrIndicator(
        active_cycle[c2], True, total_pres[c1] - total_pres[c2] <= p0_dev[c1, c2],
        name=f"Pesch0_up_c{c1}_{c2}"
    )
    modelCycle.addGenConstrIndicator(
        active_cycle[c2], True, total_pres[c2] - total_pres[c1] <= p0_dev[c1, c2],
        name=f"Pesch0_lo_c{c1}_{c2}"
    )

#### Pesch7 (Ben): equal night-shift load across cycles
recovered unchanged from Bens-Branch (9bc55cd) after the merge corruption; only a small compatibility shim was added on top.

In [32]:
# RESCUE NOTE (2026-07-08): Ben wrote this cell against the OLD pairwise Pesch2,
# which defined pair_cycles and BIG_M_HOURS. Pesch2 v2 hashed both out, so they
# are re-defined here locally to keep Ben's cell working unchanged. When Pesch7
# gets ported to the v2 share pattern (planned, see AUDIT), this shim goes away.
pair_cycles = range(1, MAX_NB_CYLCEs)                                # pairs (c, c+1)
BIG_M_HOURS = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())        # loose upper bound on cycle hours

# Pesch7: spread the NIGHT-shift load evenly across all cycles
# We measure "load" as presence time (Anwesenheitszeit = shift_hours)
# Why this compares whole cycles, not single weeks: workers rotate through the
# entire snake, so over one full turnus every worker of a cycle gets the exact
# same amount of night work. Fairness is therefore a question BETWEEN cycles
# Style: same as Pesch2 -> absolute hours, L1 deviation, soft (goes into objective).

# Night shifts are detected by clock time via overnight_ids (min-rest cell): any
# shift running past midnight counts. Stays correct when reserve clones or future
# datasets add night-type shifts, unlike the previous hard-coded class list {7, 10}.
NIGHT_IDS = set(overnight_ids)

# For every cycle c, add up the presence hours of all night shifts in it.
# We go over every week, every weekday and every shift, and only keep the shifts
# whose class is a night class. The result is one number (an expression) per cycle:
# the total night presence time of that whole snake.
night_hours = {
    c: gp.quicksum(
        shift_hours[sh] * x[c, w, d, sh]
        for w in cycleWeeks
        for d in Weekdays
        for sh in shift_hours.keys()
        if sh in NIGHT_IDS
    )
    for c in cycles
}

# Helper variables that will hold the size of the imbalance between two neighbouring
# cycles. d7_pos catches a positive gap, d7_neg a negative one. Both are >= 0.
# Their sum equals the absolute difference (that is the L1 / |...| trick).
d7_pos = modelCycle.addVars(pair_cycles, lb=0, name="pesch7_pos")
d7_neg = modelCycle.addVars(pair_cycles, lb=0, name="pesch7_neg")

# Compare each cycle with its direct neighbour (c vs c+1). Balancing every
# neighbour pair also balances the whole chain (c1 ~ c2 ~ c3 ...) by transitivity,
# exactly like Pesch2.
for c in pair_cycles:                       # pair_cycles = range(1, MAX_NB_CYLCEs), reused from Pesch2
    # difference in night hours between cycle c and the next cycle
    diff = night_hours[c] - night_hours[c + 1]

    # Force (d7_pos - d7_neg) to equal that difference, so d7_pos + d7_neg becomes |difference|.
    # The Big-M term switches the whole pair off when the next cycle is unused
    # (active_cycle = 0), so we only balance cycles that actually exist.
    modelCycle.addConstr(
        diff - (d7_pos[c] - d7_neg[c]) <=  BIG_M_HOURS * (1 - active_cycle[c + 1]),
        name=f"Pesch7_night_up_c{c}",
    )
    modelCycle.addConstr(
        diff - (d7_pos[c] - d7_neg[c]) >= -BIG_M_HOURS * (1 - active_cycle[c + 1]),
        name=f"Pesch7_night_lo_c{c}",
    )

#### Pesch6 (team 2026-07-08): equal weekend-duty load across cycles

In [33]:
# Pesch6: spread the WEEKEND duty load evenly across all cycles.
# Team decision (2026-07-08): a "weekend duty" is any working shift SCHEDULED on a
# Saturday or Sunday (calendar days 6 and 7 of the snake week), regardless of the
# shift's name or class, measured in presence time (shift_hours). This uses the
# schedule position, so weekend clones and future shift types count automatically.
# Structure: exact twin of Ben's Pesch7 above, with the night filter replaced by
# the weekend-day filter (same L1 neighbour-pair balancing, same Big-M gate).

WEEKEND_DAYS = (6, 7)  # 6 = Saturday, 7 = Sunday (see DICT_WEEKDAYS)

weekend_hours = {
    c: gp.quicksum(
        shift_hours[sh] * x[c, w, d, sh]
        for w in cycleWeeks
        for d in WEEKEND_DAYS
        for sh in shift_hours.keys()
    )
    for c in cycles
}

d6_pos = modelCycle.addVars(pair_cycles, lb=0, name="pesch6_pos")
d6_neg = modelCycle.addVars(pair_cycles, lb=0, name="pesch6_neg")

for c in pair_cycles:
    diff = weekend_hours[c] - weekend_hours[c + 1]
    modelCycle.addConstr(
        diff - (d6_pos[c] - d6_neg[c]) <=  BIG_M_HOURS * (1 - active_cycle[c + 1]),
        name=f"Pesch6_weekend_up_c{c}",
    )
    modelCycle.addConstr(
        diff - (d6_pos[c] - d6_neg[c]) >= -BIG_M_HOURS * (1 - active_cycle[c + 1]),
        name=f"Pesch6_weekend_lo_c{c}",
    )


### objective function(s)

In [34]:
print(params)
params_filtered = {k: v for k, v in params.items() if k.startswith("obj_w_")}
print(params_filtered)
print(params["obj_w_minChangeOfClasses"])
print(bool(params["obj_by_priority"]))


{'min_rest': '660', 'max_conseq_working_days': '5', 'max_cycle_length': '18', 'obj_by_priority': '0', 'obj_w_workers': '100', 'obj_w_pesch1': '30', 'obj_w_pesch2': '15', 'obj_w_minChangeOfClasses': '10', 'obj_w_pesch5': '10', 'obj_w_pesch0': '10', 'obj_w_pesch4': '10', 'avg_weekly_work_hours': '40', 'avg_reference_weeks': '4', 'max_nb_cycles': '3', 'max_weekly_work_hours': '56', 'max_free_days_per_week': '3', 'max_consec_free_days': '3', 'solver_time_limit': '300', 'obj_w_pesch7': '10', 'obj_w_pesch6': '10', 'reserve_clone_ratio': '0.2', 'solver_norel_time': '30'}
{'obj_w_workers': '100', 'obj_w_pesch1': '30', 'obj_w_pesch2': '15', 'obj_w_minChangeOfClasses': '10', 'obj_w_pesch5': '10', 'obj_w_pesch0': '10', 'obj_w_pesch4': '10', 'obj_w_pesch7': '10', 'obj_w_pesch6': '10'}
10
True


In [35]:
# Hierarchical / weighted objectives via setObjectiveN (Dirk's design, extended to ALL criteria).
# obj_by_priority = 1: priorities are derived from the weights -> Gurobi solves the
#   objectives lexicographically, highest priority first; equal weights share a tier
#   and are blended by weight within it.
# obj_by_priority = 0: all priorities are 0 -> Gurobi blends everything by weight in
#   ONE solve, which is mathematically identical to the old weighted-sum objective.
# NOTE: params values are strings, so compare against "1" (bool("0") would be True).

OBJ_BY_PRIORITY = int(params["obj_by_priority"]) == 1

objective_terms = [
    # (name, expression, weight)
    ("workers",       gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks), W_NB_WORKERS),
    ("pesch1_dev",    gp.quicksum(dev_pos[c, w] + dev_neg[c, w] for c in cycles for w in cycleWeeks), W_PESCH1),
    ("pesch2_share",  gp.quicksum(d2_pos[c, k] + d2_neg[c, k] for c in cycles for k in categories), W_PESCH2),
    ("pesch3_class",  class_changes_total, W_CHANGEofCLASSES),
    ("pesch5_type",   type_changes_total, W_PESCH5),
    ("pesch0_presence", gp.quicksum(p0_dev[c1, c2] for (c1, c2) in cycle_pairs), W_PESCH0),
    ("pesch4_spread", gp.quicksum(p4max[c] - p4min[c] for c in cycles), W_PESCH4),
    ("pesch7_night",  gp.quicksum(d7_pos[c] + d7_neg[c] for c in pair_cycles), W_PESCH7),
    ("pesch6_weekend", gp.quicksum(d6_pos[c] + d6_neg[c] for c in pair_cycles), W_PESCH6),
]

if OBJ_BY_PRIORITY:
    # lexicographic: one Gurobi objective per criterion, priority = weight
    for i, (name, expr, w) in enumerate(objective_terms):
        modelCycle.setObjectiveN(expr, index=i, priority=int(w), weight=float(w), name=name)
else:
    # blended: ONE plain objective = weighted sum of all criteria. Mathematically the
    # same as registering everything with priority 0, but Gurobi's single-objective
    # code path is much faster than the multi-objective wrapper (verified: 18 vs 34
    # active weeks found within the same 90s TimeLimit).
    modelCycle.setObjective(gp.quicksum(float(w) * expr for name, expr, w in objective_terms), GRB.MINIMIZE)

print(f"objectives: {len(objective_terms)}, mode: {'priority (lexicographic)' if OBJ_BY_PRIORITY else 'weights (blended weighted sum)'}")


objectives: 9, mode: weights (blended weighted sum)


### solver configuration

In [36]:
# Objectives are set once in the setObjectiveN cell above. Do NOT call setObjective
# here: in Gurobi it would silently redefine objective index 0 while KEEPING the
# other registered objectives, producing an unintended extra lexicographic tier.

# improvements outstanding:
    # add various weighted objectives => based on user input

# Solver settings (not part of the model, only how it is solved).
# Per-week fairness made this MIP combinatorially hard: a real objective trade-off plus
# heavy cycle/week symmetry, so proving optimality is very slow while a good solution is
# found in seconds. These params make the run practical:
#   TimeLimit (parameter solver_time_limit) -> stop and return the best solution so far;
#                     in priority mode Gurobi splits the remaining time across the tiers
#   MIPFocus  = 1   -> prioritise finding good feasible solutions over proving the bound
#   Symmetry  = 2   -> aggressively detect and discard symmetric (identical) solutions
# The large MIP gap that remains is a weak lower bound, not a bad schedule.

modelCycle.Params.TimeLimit = float(params["solver_time_limit"])
# NoRel heuristic: spend the first N seconds searching for good solutions WITHOUT
# solving the LP relaxation. Very effective here because the relaxation is weak
# (indicator constraints + Big-M), so classic node search finds incumbents slowly.
modelCycle.Params.NoRelHeurTime = float(params["solver_norel_time"])
modelCycle.Params.MIPFocus = 1
modelCycle.Params.Symmetry = 2
modelCycle.ModelSense = GRB.MINIMIZE

# warm start: reuse the previous run's solution when available. Stored as our own
# JSON of {variable name: value} because Gurobi's .sol/.mst files mangle names that
# contain [ ] % # (our shift ids) and then silently fail to match them on read.
# Matching in memory by VarName is exact; a stale/partial file is harmless.
import json as _json
WARMSTART_FILE = FOLDER_OUTPUT / "warmstart.json"
if WARMSTART_FILE.exists():
    modelCycle.update()   # flush pending vars so VarName is available
    try:
        _saved = _json.load(open(WARMSTART_FILE))
        _hits = 0
        for _v in modelCycle.getVars():
            if _v.VarName in _saved:
                _v.Start = _saved[_v.VarName]
                _hits += 1
        print(f"warm start: preloaded {_hits} of {modelCycle.NumVars} variables")
    except Exception as e:
        print(f"warm start skipped: {e}")

#run optimizer
modelCycle.optimize()

# save the incumbent for the next run's warm start
if modelCycle.SolCount > 0:
    with open(WARMSTART_FILE, "w") as _f:
        _json.dump({_v.VarName: _v.X for _v in modelCycle.getVars()}, _f)
weeks_used = sum(active_week[c, s].X for c in cycles for s in cycleWeeks)
total_dev  = sum(dev_pos[c, w].X + dev_neg[c, w].X for c in cycles for w in cycleWeeks)

# Diagnostic for the "in moeglichst vielen Turni" requirement: count how many active
# weeks hit the 40h target exactly (dev = 0). Confirms that L1 minimisation of the
# per-week deviation concentrates the unavoidable error into few weeks and leaves most
# weeks on target, so no explicit "maximise on-target weeks" constraint is needed.
active_cnt   = sum(1 for c in cycles for w in cycleWeeks if active_week[c, w].X > 0.5)
on_target    = sum(1 for c in cycles for w in cycleWeeks
                   if active_week[c, w].X > 0.5 and dev_pos[c, w].X + dev_neg[c, w].X < 1e-6)
print(f"weeks on target (dev=0): {on_target} / {active_cnt} active weeks")
print(f"active weeks: {weeks_used:.0f}, total deviation (h): {total_dev:.1f}")

# Per-objective values, mode-independent: evaluate each criterion expression on the
# incumbent solution (ObjVal alone is not meaningful for multi-objective models).
for _name, _expr, _w in objective_terms:
    print(f"objective {_name}: value {_expr.getValue():.1f} (weight {_w})")
#pesch2_imbalance = sum(d2_pos[c, k].X + d2_neg[c, k].X for c in pair_cycles for k in categories)  # OLD: pairwise metric
pesch2_imbalance = sum(d2_pos[c, k].X + d2_neg[c, k].X for c in cycles for k in categories)
print(f"Pesch2 share deviation, sum over cycles (h): {pesch2_imbalance:.1f}")
pesch8_changes = 0.5 * sum(class_diff[c, w, d, k].X for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)
print(f"Pesch8 class changes: {pesch8_changes:.0f}")
pesch5_changes = 0.5 * sum(type_diff[c, w, d, t].X for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)
print(f"Pesch5 type changes: {pesch5_changes:.0f}")
pesch0_spread = sum(p0_dev[c1, c2].X for (c1, c2) in cycle_pairs)
print(f"Pesch0 presence imbalance between cycles (h): {pesch0_spread:.1f}")
pesch4_spread = ", ".join(f"c{c}: {p4max[c].X - p4min[c].X:.1f}" for c in cycles)
print(f"Pesch4 weekly presence spread inside cycles (h): {pesch4_spread}")
pesch7_imbalance = sum(d7_pos[c].X + d7_neg[c].X for c in pair_cycles)
print(f"Pesch7 night-load imbalance between cycles (h): {pesch7_imbalance:.1f}")


Set parameter TimeLimit to value 300


Set parameter NoRelHeurTime to value 30


Set parameter MIPFocus to value 1


Set parameter Symmetry to value 2


warm start: preloaded 22526 of 22526 variables
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Pop!_OS 24.04 LTS")


...

Thread count was 4 (of 4 available processors)


Solution count 10: 5445.13 5451.97 5557.36 ... 5717.36


Time limit reached


Best objective 5.445127280441e+03, best bound 3.184970582088e+03, gap 41.5079%


weeks on target (dev=0): 8 / 22 active weeks
active weeks: 22, total deviation (h): 32.0
objective workers: value 22.0 (weight 100)
objective pesch1_dev: value 32.0 (weight 30)
objective pesch2_share: value 47.5 (weight 15)
objective pesch3_class: value 72.0 (weight 10)
objective pesch5_type: value 72.0 (weight 10)
objective pesch0_presence: value 0.0 (weight 10)
objective pesch4_spread: value 13.2 (weight 10)
objective pesch7_night: value 0.0 (weight 10)
objective pesch6_weekend: value 0.0 (weight 10)
Pesch2 share deviation, sum over cycles (h): 47.5
Pesch8 class changes: 72
Pesch5 type changes: 72
Pesch0 presence imbalance between cycles (h): 0.0
Pesch4 weekly presence spread inside cycles (h): c1: 5.2, c2: 8.0, c3: 0.0
Pesch7 night-load imbalance between cycles (h): 0.0


### results

In [37]:
# output (raw version, to be improved for better readability)
#
# FIX LOG (2026-07-07):
# 1. Filename split-brain: the header used to go to "output_cycle_.csv" (note the
#    extra underscore) while all data rows were appended to "output_cycle.csv",
#    which was never truncated. One file held only a header, the other kept
#    collecting stale rows from every previous run. Everything now goes to
#    output_cycle.csv, truncated once per run.
# 2. Python 3.10 compatibility: nested double quotes inside a double-quoted
#    f-string (f"...{shift_info[sh]["start"]}...") are a SyntaxError before
#    Python 3.12. Inner quotes are single now.
# 3. Column semantics: WorkHours = NET work time of the week (breaks excluded).
#    PresenceHours = gross presence time (Anwesenheit) of the week.
#    The class_<k>_presence_h columns split PresenceHours by shift class and sum
#    exactly to it. Reserve clones appear under their own classes (original + 1,
#    e.g. class_3 = day-week stand-by), so regular and stand-by hours are separable.
if modelCycle.SolCount > 0:   # TimeLimit -> status TIME_LIMIT, not OPTIMAL, so accept any found solution
    output_string = ""
    out_classes = sorted({dict_shift_to_class[sh] for sh in shift_hours})   # classes that carry hours
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:             # "w" truncates old runs
        file.write("created: " + str(dt.datetime.now()) + ";\n")
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;WorkHours;Deviation;OnTarget;PresenceHours;"
                   + ";".join(f"class_{k}_presence_h" for k in out_classes) + ";\n")
    for c in cycles:
        if active_cycle[c].X > 0.5:
            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                file.write(f"CYCLE: {c}:\n")
            for w in cycleWeeks:
                if active_week[c, w].X > 0.5:
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, w, d, sh].X > 0.5:
                                output_string = f"{output_string}{sh.split('%_%', 1)[0]}({shift_info[sh]['start']}-{shift_info[sh]['end']});"
                    # per-week net work hours and deviation from the 40h target (from Pesch1 dev vars)
                    signed_dev = dev_pos[c, w].X - dev_neg[c, w].X
                    week_h     = AVG_WEEKLY_HOURS + signed_dev
                    on_target  = "yes" if abs(signed_dev) < 1e-6 else "no"
                    output_string += f"{week_h:.1f};{signed_dev:+.1f};{on_target};"
                    # gross presence of the week: the reference total for the class columns
                    presence_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                     for d in Weekdays for sh in shift_hours)
                    output_string += f"{presence_h:.1f};"
                    # presence hours split by shift class (columns sum to PresenceHours)
                    for k in out_classes:
                        class_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                      for d in Weekdays for sh in shift_hours if dict_shift_to_class[sh] == k)
                        output_string += f"{class_h:.1f};"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""